In [1]:
!pip install colorthief piexif

In [7]:
!unzip /content/flickr.zip

Archive:  /content/flickr.zip
   creating: flickr/
  inflating: flickr/-_3035910359_o.jpg  
  inflating: flickr/-_3036742962_o.jpg  
  inflating: flickr/-_3559411717_o.jpg  
  inflating: flickr/.BridgeLabelsAndRatings  
   creating: flickr/.ipynb_checkpoints/
  inflating: flickr/0183_3172274394_o.jpg  
  inflating: flickr/0191_3172276864_o.jpg  
  inflating: flickr/0570_3430754948_o.jpg  
  inflating: flickr/0574_3430300019_o.jpg  
  inflating: flickr/0577_3431112728_o.jpg  
  inflating: flickr/0580_3430301965_o.jpg  
  inflating: flickr/0581_3430302279_o.jpg  
  inflating: flickr/101-aniversario-gabriel-figueroa_2476343889_o.jpg  
  inflating: flickr/111_3036809310_o.jpg  
  inflating: flickr/1559963552_6261a7c22b_o.jpg  
  inflating: flickr/2640117592_ae14818748_b_3860942908_o.jpg  
  inflating: flickr/3035690897_8d33a0daa0_o.jpg  
  inflating: flickr/3036044871_8439970fe2_o.jpg  
  inflating: flickr/3036526788_cd88b41b13_o.jpg  
  inflating: flickr/3036528336_864365a3a6_o.jpg  
  in

In [ ]:
from colorthief import ColorThief

color_thief = ColorThief('soy.jpg')
# get the dominant color
dominant_color = color_thief.get_color(quality=1)
print (dominant_color)
# get a palette of dominant colors
palette = color_thief.get_palette(color_count=5)
print (palette)


(182, 177, 169)
[(182, 177, 169), (30, 33, 33), (92, 96, 96), (73, 64, 74), (87, 84, 60)]


### Processing a folder of JPG files

Now, let's create a function that can iterate through a given folder, find all `.jpg` (or `.jpeg`) files, extract their dominant color, and then save this color to the image's metadata using the `save_dominant_color_to_metadata` function.

In [8]:
import os
import csv
from colorthief import ColorThief

def process_folder_for_dominant_color(folder_path):
    if not os.path.isdir(folder_path):
        print(f"Error: Folder '{folder_path}' not found.")
        return

    # Dynamically set CSV file name based on folder_path
    csv_file_name = f'{folder_path}_color_palettes.csv'
    # Check if CSV file exists to write header, or create it if it doesn't
    file_exists = os.path.isfile(csv_file_name)

    # Open CSV file in append mode, create if not exists
    with open(csv_file_name, 'a', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        if not file_exists:
            csv_writer.writerow(['filename', 'color_palette'])

        print(f"Processing images in folder: {folder_path}")
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.jpg', '.jpeg')):
                image_path = os.path.join(folder_path, filename)
                try:
                    print(f"  Processing {filename}...")
                    color_thief = ColorThief(image_path)
                    palette = color_thief.get_palette(color_count=5) # Extract palette as requested

                    # Write filename and palette to CSV
                    csv_writer.writerow([filename, str(palette)])

                    print(f"Palette for '{filename}' saved to '{csv_file_name}'")
                except Exception as e:
                    print(f"  Error processing {filename}: {e}")
            else:
                print(f"  Skipping non-JPG file: {filename}")
    print("Folder processing complete.")

### Demonstration of folder processing

To demonstrate this function, let's first create a dummy folder and copy the `soy.jpg` file into it multiple times with different names. Then, we will call `process_folder_for_dominant_color` on this new folder.

In [9]:
# Create a dummy folder for demonstration
demo_folder = 'flickr'
os.makedirs(demo_folder, exist_ok=True)

# Now, call the new function to process the folder
process_folder_for_dominant_color(demo_folder)

Processing images in folder: flickr
  Processing con-olympus-pen-ft_4704632053_o.jpg...
Palette for 'con-olympus-pen-ft_4704632053_o.jpg' saved to 'flickr_color_palettes.csv'
  Processing el-rally_4439997886_o.jpg...
Palette for 'el-rally_4439997886_o.jpg' saved to 'flickr_color_palettes.csv'
  Processing ciudad-universitaria-3_2640117592_o.jpg...
Palette for 'ciudad-universitaria-3_2640117592_o.jpg' saved to 'flickr_color_palettes.csv'
  Processing auto-verde_2770467139_o.jpg...
Palette for 'auto-verde_2770467139_o.jpg' saved to 'flickr_color_palettes.csv'
  Processing img_0185_12040600754_o.jpg...
Palette for 'img_0185_12040600754_o.jpg' saved to 'flickr_color_palettes.csv'
  Processing vaca_1359780211_o.jpg...
Palette for 'vaca_1359780211_o.jpg' saved to 'flickr_color_palettes.csv'
  Processing img092a_7168518858_o.jpg...
Palette for 'img092a_7168518858_o.jpg' saved to 'flickr_color_palettes.csv'
  Processing afilador_3560104602_o.jpg...
Palette for 'afilador_3560104602_o.jpg' saved

### Generate HTML Visualization File

Now, let's create the `color_distribution.html` file. This file will contain embedded CSS for styling and JavaScript to dynamically render the image cards with their dominant colors and palettes based on the `image_color_data` we just collected.

In [13]:
import os
import csv
import json
import ast # To safely evaluate the string representation of the palette

image_color_data = []
demo_folder = 'flickr' # Ensure this matches the folder processed by process_folder_for_dominant_color
csv_file_name = f'{demo_folder}_color_palettes.csv'

if os.path.exists(csv_file_name):
    with open(csv_file_name, 'r', newline='') as csvfile:
        csv_reader = csv.reader(csvfile)
        header = next(csv_reader) # Skip header row
        for row in csv_reader:
            if len(row) == 2:
                filename = row[0]
                palette_str = row[1]
                try:
                    # Safely evaluate the string representation of the palette tuple
                    palette = ast.literal_eval(palette_str)
                    if palette and isinstance(palette, list) and len(palette) > 0:
                        dominant_color = palette[0] # Assume the first color in the palette is the dominant one for display
                        image_color_data.append({
                            'filename': filename,
                            'path': os.path.join(demo_folder, filename), # Construct the image path
                            'dominant_color': dominant_color,
                            'palette': palette
                        })
                    else:
                        print(f"Skipping {filename}: Invalid or empty palette data.")
                except (ValueError, SyntaxError) as e:
                    print(f"Error parsing palette for {filename}: {e}")
            else:
                print(f"Skipping malformed row in CSV: {row}")
    print(f"Loaded {len(image_color_data)} image color entries from {csv_file_name}.")
else:
    print(f"CSV file '{csv_file_name}' not found. Cannot generate HTML visualization.")


html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Image Color Distribution</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f4f4f4;
            color: #333;
        }}
        h1 {{
            text-align: center;
            color: #0056b3;
        }}
        #image-container {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
            gap: 20px;
            padding: 20px;
            justify-content: center;
        }}
        .image-card {{
            background-color: #fff;
            border-radius: 8px;
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
            overflow: hidden;
            text-align: center;
            transition: transform 0.2s;
        }}
        .image-card:hover {{
            transform: translateY(-5px);
        }}
        .image-card img {{
            max-width: 100%;
            height: 200px; /* Fixed height for consistency */
            object-fit: cover; /* Cover the area, cropping if necessary */
            display: block;
            margin: 0 auto;
            border-bottom: 1px solid #eee;
        }}
        .image-info {{
            padding: 15px;
        }}
        .image-info h3 {{
            margin-top: 0;
            margin-bottom: 10px;
            color: #0056b3;
        }}
        .color-display {{
            display: flex;
            justify-content: center;
            margin-top: 10px;
            flex-wrap: wrap;
        }}
        .color-swatch {{
            width: 40px;
            height: 40px;
            margin: 5px;
            border: 1px solid #ddd;
            border-radius: 4px;
            display: flex;
            align-items: center;
            justify-content: center;
            color: white;
            font-size: 0.7em;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.8);
        }}
        .dominant-color-label {{
            font-weight: bold;
            margin-top: 10px;
            display: block;
        }}
    </style>
</head>
<body>
    <h1>Image Color Distribution</h1>
    <div id="image-container"></div>

    <script>
        const imageData = {json.dumps(image_color_data)};
        const imageContainer = document.getElementById('image-container');

        imageData.forEach(item => {{
            const card = document.createElement('div');
            card.className = 'image-card';

            const img = document.createElement('img');
            img.src = item.path; // Assuming path is relative or accessible
            img.alt = item.filename;
            card.appendChild(img);

            const info = document.createElement('div');
            info.className = 'image-info';
            info.innerHTML = '<h3>' + item.filename + '</h3>';

            const dominantColorLabel = document.createElement('span');
            dominantColorLabel.className = 'dominant-color-label';
            dominantColorLabel.textContent = 'Dominant Color:';
            info.appendChild(dominantColorLabel);

            const dominantColorSwatch = document.createElement('div');
            dominantColorSwatch.className = 'color-swatch';
            // Ensure dominant_color exists, otherwise use a default or handle gracefully
            if (item.dominant_color) {{
                dominantColorSwatch.style.backgroundColor = 'rgb(' + item.dominant_color[0] + ', ' + item.dominant_color[1] + ', ' + item.dominant_color[2] + ')';
                dominantColorSwatch.title = 'RGB(' + item.dominant_color[0] + ', ' + item.dominant_color[1] + ', ' + item.dominant_color[2] + ')';
            }} else {{
                dominantColorSwatch.style.backgroundColor = '#cccccc'; // Default grey
                dominantColorSwatch.title = 'No dominant color';
            }}
            info.appendChild(dominantColorSwatch);

            const paletteLabel = document.createElement('span');
            paletteLabel.className = 'dominant-color-label';
            paletteLabel.textContent = 'Palette:';
            info.appendChild(paletteLabel);

            const paletteDisplay = document.createElement('div');
            paletteDisplay.className = 'color-display';
            item.palette.forEach(color => {{
                const swatch = document.createElement('div');
                swatch.className = 'color-swatch';
                swatch.style.backgroundColor = 'rgb(' + color[0] + ', ' + color[1] + ', ' + color[2] + ')';
                swatch.title = 'RGB(' + color[0] + ', ' + color[1] + ', ' + color[2] + ')';
                paletteDisplay.appendChild(swatch);
            }});
            info.appendChild(paletteDisplay);

            card.appendChild(info);
            imageContainer.appendChild(card);
        }});
    </script>
</body>
</html>
"""

with open('color_distribution.html', 'w') as f:
    f.write(html_content)

print("Generated color_distribution.html")

Loaded 371 image color entries from flickr_color_palettes.csv.
Generated color_distribution.html


### View the HTML File

After executing the previous cell, a file named `color_distribution.html` will be created in your Colab environment. You can download this file and open it in a web browser to view the visualization.

In [16]:
!npm install -g localtunnel

import http.server
import socketserver
import os
import threading
import time
import subprocess

PORT = 8000
DIRECTORY = "."

class MyHandler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=DIRECTORY, **kwargs)

def start_server():
    # Ensure the HTML file is in the current directory or specified directory
    if not os.path.exists(os.path.join(DIRECTORY, 'color_distribution.html')):
        print(f"Error: 'color_distribution.html' not found in {DIRECTORY}.")
        return

    # Use ThreadingTCPServer to allow multiple requests and prevent blocking Colab
    with socketserver.ThreadingTCPServer(("", PORT), MyHandler) as httpd:
        print(f"Serving files from '{DIRECTORY}' on port {PORT}")
        try:
            httpd.serve_forever()
        except KeyboardInterrupt:
            print("HTTP Server stopped.")

def start_localtunnel_and_server():
    # Start the HTTP server in a separate thread
    server_thread = threading.Thread(target=start_server)
    server_thread.daemon = True
    server_thread.start()

    # Give the HTTP server a moment to start
    time.sleep(2)

    try:
        print("Starting localtunnel...")
        # Use subprocess to run localtunnel command
        # The --port argument tells localtunnel which local port to expose
        # The output of localtunnel contains the public URL
        lt_process = subprocess.Popen(['lt', '--port', str(PORT)], stdout=subprocess.PIPE, text=True)

        # Read output from localtunnel process to find the URL
        public_url = None
        for line in iter(lt_process.stdout.readline, ''):
            print(line.strip())
            if "your url is:" in line:
                public_url = line.split("your url is:")[1].strip()
                break

        if public_url:
            print(f"\nPublic URL for 'color_distribution.html': {public_url}/color_distribution.html")
            print("To stop the server and localtunnel, interrupt this cell's execution.")

            # Keep the main thread alive while localtunnel is running
            while lt_process.poll() is None:
                time.sleep(1)
        else:
            print("Could not get public URL from localtunnel.")

    except Exception as e:
        print(f"An error occurred with localtunnel: {e}")
        print("Please ensure 'localtunnel' is properly installed and accessible.")
    finally:
        if 'lt_process' in locals() and lt_process.poll() is None:
            lt_process.terminate()
            print("localtunnel process terminated.")
        print("Server process finished.")

# Execute the function to start localtunnel and the server
start_localtunnel_and_server()

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 2s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

Exception in thread Thread-6 (start_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_858/202557832.py", line 24, in start_server
  File "/usr/lib/python3.12/socketserver.py", line 457, in __init__
    self.server_bind()
  File "/usr/lib/python3.12/socketserver.py", line 478, in server_bind
    self.socket.bind(self.server_address)
OSError: [Errno 98] Address already in use


Starting localtunnel...
your url is: https://sharp-clouds-brake.loca.lt

Public URL for 'color_distribution.html': https://sharp-clouds-brake.loca.lt/color_distribution.html
To stop the server and localtunnel, interrupt this cell's execution.


127.0.0.1 - - [22/May/2026 00:39:41] "GET /color_distribution.html HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:41] "GET /flickr/con-olympus-pen-ft_4704632053_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:42] "GET /flickr/el-rally_4439997886_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:42] "GET /flickr/ciudad-universitaria-3_2640117592_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:42] "GET /flickr/auto-verde_2770467139_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:42] "GET /flickr/img092a_7168518858_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:42] "GET /flickr/vaca_1359780211_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:42] "GET /flickr/img_0185_12040600754_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:42] "GET /flickr/afilador_3560104602_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:43] "GET /flickr/muro_5426717940_o.jpg HTTP/1.1" 200 -
127.0.0.1 - - [22/May/2026 00:39:43] "GET /flickr/anden-del-metro-patri

localtunnel process terminated.
Server process finished.


KeyboardInterrupt: 